In [ ]:
from pathlib import Path
import pandas as pd

STATIC_PARAMETERS = {"RecordID", "Age", "Gender", "Height", "ICUType", "Weight"}
STATIC_EXCLUDE = {"RecordID", "Age", "Gender", "Height", "ICUType"}

def hhmm_to_minutes(value: str) -> int:
    hours, minutes = value.split(":")
    return int(hours) * 60 + int(minutes)

def parse_patient_file(file_path: Path):
    df = pd.read_csv(file_path)
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    record_rows = df.loc[df["Parameter"] == "RecordID", "Value"].dropna()
    patient_id = int(record_rows.iloc[0]) if not record_rows.empty else int(file_path.stem)

    static_df = df[df["Parameter"].isin(STATIC_PARAMETERS)].copy()
    static_df = static_df.drop_duplicates(subset=["Parameter"], keep="last")
    static_values = static_df.set_index("Parameter")["Value"].to_dict()

    static_record = {
        "RecordID": patient_id,
        "Age": static_values.get("Age"),
        "Gender": static_values.get("Gender"),
        "Height": static_values.get("Height"),
        "Weight": static_values.get("Weight"),
        "ICUType": static_values.get("ICUType"),
    }

    dynamic_df = df[~df["Parameter"].isin(STATIC_EXCLUDE)].copy()
    dynamic_df["Minutes"] = dynamic_df["Time"].map(hhmm_to_minutes)
    dynamic_df["RecordID"] = patient_id
    dynamic_df = dynamic_df[["RecordID", "Time", "Minutes", "Parameter", "Value"]]

    return static_record, dynamic_df

def load_cohort(folder: str):
    folder_path = Path(folder)
    static_records = []
    dynamic_tables = []

    for file_path in sorted(folder_path.glob("*.txt")):
        static_record, dynamic_df = parse_patient_file(file_path)
        static_records.append(static_record)
        dynamic_tables.append(dynamic_df)

    patients_static = pd.DataFrame(static_records).drop_duplicates(subset=["RecordID"])
    patients_static = patients_static.set_index("RecordID").sort_index()

    if dynamic_tables:
        patient_events = pd.concat(dynamic_tables, ignore_index=True)
        patient_events = patient_events.sort_values(["RecordID", "Minutes", "Parameter"]).reset_index(drop=True)
    else:
        patient_events = pd.DataFrame(columns=["RecordID", "Time", "Minutes", "Parameter", "Value"])

    return patients_static, patient_events

In [ ]:
train_data_location = "set-a"
patients_static, patient_events = load_cohort(train_data_location)

print(f"Number of patients: {patients_static.shape[0]}")
print(f"Number of dynamic observations: {patient_events.shape[0]}")

display(patients_static.head())
display(patient_events.head(15))

In [ ]:
#create a pivot table of patient_events
pivot_table = patient_events.pivot_table(index=["RecordID", "Time", "Minutes"], columns="Parameter", values="Value")
print("Pivot table of patient_events:")
# display(pivot_table.head(5))
#find how many columns are in the pivot table
print(f"Number of columns in the pivot table: {pivot_table.shape[1]}")
#make it a dataframe again
pivot_table = pivot_table.reset_index()
print("Pivot table of patient_events after resetting index:")
# display(pivot_table.head(5))
#join the static variables to the pivot table
pivot_table = pivot_table.merge(patients_static, on="RecordID", how="left")
pivot_table = pivot_table.sort_values(["RecordID", "Minutes"]).reset_index(drop=True)
print("Pivot table of patient_events after merging with patients_static:")
# display(pivot_table.head(5))
#values that gender variable can take
print(f"Unique values in Gender column: {pivot_table['Gender'].unique()}")
#minus one for the 'Gender' 0,1,-1
print(f"Number of columns in the pivot table after merging with patients_static: {pivot_table.shape[1]}")
print(f"pivot table columns: {pivot_table.columns.tolist()}")

In [ ]:
pivot_table

In [ ]:
#find unique parameters in patient_events
unique_parameters = patient_events["Parameter"].unique()
print(f"Unique parameters in patient_events: {unique_parameters}")
print(f"length of unique parameters: {len(unique_parameters)}")

In [ ]:
# load in outcomes for the training set (set-a)
outcomes = pd.read_csv("Outcomes-a.txt")
print(f"Number of outcome records: {outcomes.shape[0]}")
#number of null records
num_null_records = outcomes['In-hospital_death'].isnull().sum()
print(f"Number of null records in 'In-hospital_death': {num_null_records}")
outcomes#[['In-hospital_death']]